#### Librerías

In [ ]:
import pandas as pd

In [ ]:
import openpyxl

In [ ]:
import datetime as dt

In [ ]:
import numpy as np

In [ ]:
import win32com.client as win32

In [ ]:
import os, shutil

In [ ]:
import win32con, win32api

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import logging

#### Vaciar la carpeta V:/BIVSA/ADMOPER/Cierre diario/Control

In [ ]:
vaciar = 'V:/BIVSA/ADMOPER/Cierre diario/Control'
for files in os.listdir(vaciar):
    path = os.path.join(vaciar, files)
    try:
        shutil.rmtree(path)
    except OSError:
        os.remove(path) 

#### Vaciar la carpeta de las Curvas

In [ ]:
vaciar = 'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/'
for files in os.listdir(vaciar):
    path = os.path.join(vaciar, files)
    try:
        shutil.rmtree(path)
    except OSError:
        os.remove(path) 

#### Definición de variables de fechas

In [ ]:
hoy = dt.date.today() #Determina fecha del día.

In [ ]:
fa=(hoy.strftime('%Y%m%d')) #Determina la fecha del día par la ruta del archivo.

#### Definiciones para el archivo de Log

In [ ]:
logger = logging.getLogger() #Llama a la función de logging.

In [ ]:
logger.setLevel(logging.INFO) #Define el nivel a partir del que se mostrarán los mensajes.

In [ ]:
log_local = 'C:/Temp/control_diario.log' #Archivo donde guarda el log localmente para evitar problemas de red.

In [ ]:
log_red = 'V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa +'/control_diario.log' #Archivo final en la red.

In [ ]:
fhandler = logging.FileHandler(filename=log_local, mode='a') #Archivo donde guarda el log.

In [ ]:
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s') #Formato de las líneas de log.

In [ ]:
fhandler.setFormatter(formatter) #Setea los formatos.

In [ ]:
logger.addHandler(fhandler) #Agrega los formatos.

#### Usuarios permitidos

In [ ]:
usuarios = ('sm113925','cs214857','cr118369','ia214828')

#### Función Inicio control

In [ ]:
def inicio_control():
    #Información para el archivo de log------------------------------------------------------------
    logging.info('----Control de precios Version 3.1 - Actualizado: 06/10/2025----') #Mensaje para el logging.
    logging.info('Usuario operador: ' + user) #Mensaje para el logging. 
    logging.info('Información descargada: ' + descarga) #Mensaje para el logging. 
    logging.info('Se inicia el proceso de generación de información para el control de precios.') #Mensaje para el logging.   
    print('🧩 Iniciando proceso de consolidación de información...') #Mensaje a mostrar en la consola.

#### Archivos para verificar actualizaciones

In [ ]:
def actualizaciones():
    archivos = [
        'V:/BIVSA/ADMOPER/Cierre diario/VF/Cobro de cupones.xlsx',
        'V:/BIVSA/ADMOPER/Cierre diario/VF/Custodia RFG.xlsx',
        'V:/BIVSA/ADMOPER/Cierre diario/VF/Cotizaciones de papeles.xlsx'
    ]

    def obtener_ultima_actualizacion(ruta_archivo):
        try:
            timestamp = os.path.getmtime(ruta_archivo)
            fecha_hora = dt.datetime.fromtimestamp(timestamp)
            return fecha_hora.strftime("%Y-%m-%d %H:%M:%S")
        except FileNotFoundError:
            return "¡Archivo no encontrado!"
        except Exception as e:
            return f"Error: {str(e)}"

    for archivo in archivos:
        ultima_modificacion = obtener_ultima_actualizacion(archivo)
        print(f"⚠️ Archivo {archivo} actualizado el {ultima_modificacion}")
        #print(f"Última modificación: {ultima_modificacion}\n")
        logging.info(f"Archivo {archivo} actualizado el {ultima_modificacion}") #Mensaje para el logging. 

#### Cuadros de curvas

In [ ]:
def curvas():
    # Mostrar todas las columnas y sin truncar el ancho
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width',          None)

    #print(f"{'*' * 40}\n         {'Curvas de Renta Fija'}\n{'*' * 40}")
    #print('\n')

    # Importación de archivos
    try:
        equivalencias = pd.read_excel(r'R:\Files\User\PORTFOLI\PORFOLIO\RentaFija\Vectores\Curvas Renta Fija\Equivalencias_Curvas.xlsx', sheet_name='Equivalencias')
        curva = pd.read_excel('V:/BIVSA/ADMOPER/Cierre diario/Vectores/' + fa + ' CAFCI Vectores - Precios C.xlsx', skiprows=1) #Crea la ruta

    except FileNotFoundError as e:
        print(f"Error: {e}")
        raise

    # Asignación de Curvas
    equivalencias['Curva'] = np.nan

    map_curvas = {
        'BADLAR FF - FF Tier 1': 'Badlar_TAMAR FF',
        'BADLAR FF - FF Tier 2': 'Badlar_TAMAR FF',
        'BADLAR FF - FF Tier 3': 'Badlar_TAMAR FF',
        'BADLAR FF - FF PyME': 'Badlar_TAMAR FF',
        'BADLAR FF - FF PyME': 'Badlar_TAMAR FF',
        'TAMAR FF - FF Tier 1': 'Badlar_TAMAR FF',
        'TAMAR FF - FF Tier 2': 'Badlar_TAMAR FF',
        'TAMAR FF - FF Tier 3': 'Badlar_TAMAR FF',
        
        'BADLAR ON - ON PyME Gar': 'Badlar ON Pyme',
        
        'BADLAR ON - ON Tier 1': 'ON ARS BADLAR&TAMAR',
        'BADLAR ON - ON Tier 2': 'ON ARS BADLAR&TAMAR',
        'BADLAR ON - ON Tier 3': 'ON ARS BADLAR&TAMAR',
        'BADLAR ON - ON PyME Gar': 'ON ARS BADLAR&TAMAR',
        'TAMAR ON - ON Tier 1': 'ON ARS BADLAR&TAMAR',
        'TAMAR ON - ON Tier 2': 'ON ARS BADLAR&TAMAR',
        'TAMAR ON - ON Tier 3': 'ON ARS BADLAR&TAMAR',
        'TAMAR ON - ON PyME Gar': 'ON ARS BADLAR&TAMAR',

        'Tasa Fija Pesos ON - ON Tier 1': 'Tasa Fija ON',
        'Tasa Fija Pesos ON - ON Tier 2': 'Tasa Fija ON',
        'Tasa Fija Pesos ON - ON Tier 3': 'Tasa Fija ON',
        
        'Tasa Fija Pesos FF - ON Tier 1': 'Tasa Fija FF',
        'Tasa Fija Pesos FF - ON Tier 2': 'Tasa Fija FF',
        'Tasa Fija Pesos FF - ON Tier 3': 'Tasa Fija FF',
        
        'Tasa Fija Pesos CER - ON Tier 1 UVA': 'ON CER',
        'Tasa Fija Pesos CER - ON Tier 2 UVA': 'ON CER',
        
        'Tasa Fija USD Ley Extranjera ON - ON Tier 1': 'ON HD Ley Ext',
        'Tasa Fija USD Ley Extranjera ON - ON Tier 2': 'ON HD Ley Ext',
        'Tasa Fija USD Ley Extranjera ON - ON Tier 3': 'ON HD Ley Ext',
        
        'Tasa Fija USD Ley Argentina ON - ON Tier 1': 'ON HD Ley Arg',
        'Tasa Fija USD Ley Argentina ON - ON Tier 2': 'ON HD Ley Arg',
        'Tasa Fija USD Ley Argentina ON - ON Tier 3': 'ON HD Ley Arg',
        
        'Tasa Fija USD Link ON - ON Tier 1': 'DL ON',
        'Tasa Fija USD Link ON - ON Tier 2': 'DL ON',
        'Tasa Fija USD Link ON - ON Tier 3': 'DL ON',
        'Z-Spread D': 'Z-Spread D',
        
        'Tasa Fija USD Link ON - ON PyME Gar': 'DL ON PyME',
        'Tasa Fija USD Link ON - ON PyME': 'DL ON PyME',

        'Tasa Fija USD Ley Extranjera - Soberano': 'Soberano HD',
        'Tasa Fija USD Ley Argentina - Soberano': 'Soberano HD',

        'Tasa Fija Pesos CER - Soberano': 'Soberano CER'        
    }

    equivalencias['Curva'] = equivalencias['Tipo de activo'].map(map_curvas)
    
    output_dir=r'R:\Files\User\PORTFOLI\PORFOLIO\RentaFija\Vectores\Curvas Renta Fija\outputs'
    #output_dir='V:/BIVSA/ADMOPER/Cierre diario/Out-curvas'
    os.makedirs(output_dir, exist_ok=True)

    curva = curva[['BYMA', 'TIR [%]', 'Mod. Duration', 'Spread [%]']]
    curva = curva.dropna(subset=['BYMA', 'Mod. Duration', 'TIR [%]'])
    df = pd.merge(curva, equivalencias[['BYMA', 'Curva', 'alpha']], on='BYMA', how='left')
    df = df.dropna(subset=['Curva'])
    df = df[df['Curva'] != np.nan]


    df['Mod.Duration Cdo'] = df['Mod. Duration'].round(2)
    df['TIR Cdo [%]'] = df['TIR [%]'].round(2)
    df['Spread Cdo. [%]'] = df['Spread [%]'].round(2)

    # Grafico de curvas
    for curva_name in df['Curva'].unique():

        fig, ax = plt.subplots(figsize=(12, 6))
        fig.patch.set_facecolor('black')
        ax.set_facecolor('black')

        base_color = 'cyan'
        filtered_df = df[df['Curva'] == curva_name].copy()
        
        y_column = 'Spread Cdo. [%]' if curva_name == 'ON ARS BADLAR&TAMAR' else 'TIR Cdo [%]'
        y_label = 'Spread (%)' if y_column == 'Spread Cdo. [%]' else 'TIR (%)'


        if 'alpha' not in filtered_df.columns:
            filtered_df['alpha'] = 0

        mask_alpha = (filtered_df['alpha'] == 1)
        df_alpha = filtered_df[mask_alpha]
        df_no_alpha = filtered_df[~mask_alpha]


        if not df_no_alpha.empty:
            ax.scatter(df_no_alpha['Mod.Duration Cdo'], df_no_alpha[y_column],s=50, c=base_color, alpha=0.4, label='Data Points')

        # Puntos con alpha=1 en rojo
        if not df_alpha.empty:
            ax.scatter(df_alpha['Mod.Duration Cdo'], df_alpha[y_column],s=70, c='red', alpha=0.8, label='Alpha')

        # Etiquetas BYMA (rojas si alpha=1)
        for _, row in filtered_df.iterrows():
            txt_color = 'red' if row.get('alpha', 0) == 1 else 'white'
            ax.text(row['Mod.Duration Cdo'], row[y_column], row['BYMA'],fontsize=6, ha='right', color=txt_color)

        # Ajuste logarítmico
        x_vals = filtered_df['Mod.Duration Cdo']
        y_vals = filtered_df[y_column]
        valid_x = x_vals[x_vals > 0]
        valid_y = y_vals.loc[valid_x.index]
        if len(valid_x) > 1:
            fit = np.polyfit(np.log(valid_x), valid_y, 1)
            x_fit = np.linspace(min(valid_x), max(valid_x), 100)
            y_fit = fit[0] * np.log(x_fit) + fit[1]
            ax.plot(x_fit, y_fit, color='white', linewidth=3, linestyle='-', label='Ajuste Log')

        ax.set_title(f'Curva de Renta Fija {curva_name}', fontsize=10, color='white')
        ax.set_xlabel('Mod. Duration', fontsize=12, color='white')
        ax.set_ylabel(y_label, fontsize=8, color='white')

        ax.tick_params(axis='x', colors='white')
        ax.tick_params(axis='y', colors='white')

        ax.grid(True, linestyle='--', alpha=0.5, color='gray')
        ax.legend(facecolor='darkgray', edgecolor='white', fontsize=8)


        plt.tight_layout()

        output_file = os.path.join(output_dir, f"Curva_{curva_name}.jpg")
        plt.savefig(output_file, format='jpg', dpi=300, bbox_inches='tight')
        plt.close()

        print(f"📈 Gráfico de curva guardado en: {output_file}")

        # ====================== Lista segmentada por Curva: solo alpha=1 ======================

        df_alpha = df[df['alpha'] == 1].dropna(subset=['BYMA', 'Curva'])

        activos_por_curva = (
            df_alpha[['Curva', 'BYMA']]
            .drop_duplicates()
            .sort_values(['Curva', 'BYMA'])
            .groupby('Curva')['BYMA']
            .apply(list)
        )

        n_curvas = len(activos_por_curva)
        max_activos = max(len(lst) for lst in activos_por_curva) if n_curvas > 0 else 1

        fig_w = min(22, 5 + 3.0 * n_curvas)
        fig_h = min(26, 2 + 0.35 * max_activos)

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        fig.patch.set_facecolor('black')
        ax.set_facecolor('black')
        ax.axis('off')


        ax.text(0.5, 1.02,
                f"Lista de activos con tenencia ({df_alpha['BYMA'].nunique()} activos)",
                ha='center', va='bottom', color='white', fontsize=14, weight='bold',
                transform=ax.transAxes)


        col_width = 1.0 / max(1, n_curvas)
        row_height = 1.0 / (max_activos + 2)

        fs = 9

        for j, (curva, lista_activos) in enumerate(activos_por_curva.items()):

            x_base = j * col_width + 0.01


            ax.text(x_base + col_width/2, 0.98,
                    curva,
                    color='white', fontsize=11, ha='center', va='top',
                    transform=ax.transAxes)


            for i, ticker in enumerate(lista_activos):
                y = 0.95 - (i+1) * row_height
                ax.text(x_base, y, f"• {ticker}",
                        color='white', fontsize=fs, ha='left', va='top',
                        transform=ax.transAxes)

        lista_alpha_path = os.path.join(output_dir, "Lista_Activos_Alpha.png")
        plt.tight_layout()
        plt.savefig(lista_alpha_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
        plt.close(fig)
        
        logging.info(f"Curvas de renta fija generadas exitosamente en {output_file}") #Mensaje para el logging. 

#### Arma la tabla de control

In [ ]:
def tabla_control():
    #Importa las precios de VF
    cotizaciones = pd.read_excel ('V:/BIVSA/ADMOPER/Cierre diario/VF/Cotizaciones de papeles.xlsx', skiprows=[0,2], engine='openpyxl') #Importa los datos del archivo.

    #Importa el archivo de CAFCI Precios C
    file = 'V:/BIVSA/ADMOPER/Cierre diario/Vectores/' + fa + ' CAFCI Vectores - Precios C.xlsx' #Crea la ruta.
    precios_c = pd.read_excel (file, skiprows=[0], engine='openpyxl') #Importa los datos del archivo.
    precios_c = precios_c.iloc[:, [2, 21]] #Seleeciona las columnas con las que se queda.
    precios_c = precios_c.rename (columns={'BYMA':'Abreviatura','48 hs..1':'Criterio'}) #Renombra las columnas.

    #Importa el archivo de CAFCI Cortes de cupón
    dia = pd.Timestamp.today().date() #Determina fecha para el cálculo de Días restantes.
    cortes = pd.read_excel ('V:/BIVSA/ADMOPER/Cierre diario/Vectores/' + fa + ' CAFCI Vectores - Corte de Cupon.xlsx', skiprows=[0], engine='openpyxl') #Importa los datos del archivo.
    cortes = cortes.iloc[:, [2, 22, 26]] #Seleeciona las columnas con las que se queda.
    cortes = cortes.rename (columns={'BYMA':'Abreviatura','Fecha de Pago':'Próximo Cupón','Total Pagado':'Total Cupón (R+A)'}) #Renombra las columnas.
    cortes['Días restantes'] = cortes['Próximo Cupón'].apply(lambda x: np.busday_count(dia, x.date())) #Crea y calcula la columna de días restantes
    cortes = cortes[['Abreviatura', 'Próximo Cupón', 'Días restantes', 'Total Cupón (R+A)']] #Ordena las columnas.

    #Importa el archivo de Cobros
    cobros = pd.read_excel ('V:/BIVSA/ADMOPER/Cierre diario/VF/Cobro de cupones.xlsx', skiprows=[0,2], engine='openpyxl') #Importa los datos del archivo.
    cobros = cobros.drop(cobros.columns[[0,2,3,5]], axis=1) #Elimina las columnas que nos necesita.
    cobros = cobros.rename (columns={'Especie.1':'Descripción','Fecha':'Cupón cobrado'}) #Renombra las columnas.

    #Importa el archivo de Custodia RGF
    rfg = pd.read_excel ('V:/BIVSA/ADMOPER/Cierre diario/VF/Custodia RFG.xlsx', skiprows=[0,1,3], engine='openpyxl') #Importa los datos del archivo.
    rfg = rfg.drop(rfg.columns[[0,2,3,4,5,6]], axis=1) #Elimina las columnas que no necesita.
    pos_columna_a_insertar = rfg.columns.get_loc('Papel') #Busca la ubicación de la columna 1, para tener de referencia.
    rfg.insert(pos_columna_a_insertar + 1, 'Tenencia', 'Con tenencia') #Inserta la columna Tenencia.
    rfg = rfg.rename (columns={'Papel':'Abreviatura', 'Tenencia':'RFG'}) #Renombra las columnas.

    #Arma la tabla de control
    control = cotizaciones.merge(precios_c, on='Abreviatura',how='left').merge(cortes, on='Abreviatura',how='left').merge(cobros, on='Descripción',how='left').merge(rfg, on='Abreviatura',how='left') #Une las tres tablas.
    control = control.sort_values('Variación (%)', ascending=True) #Ordena por variación.
    control = control.drop_duplicates() #Elimina registros duplicados.

    return(control)

#### Exportación de archivo

In [ ]:
def export():
    control = tabla_control() #DataFerame de la tabla de control
    control['Próximo Cupón'] = pd.to_datetime(control['Próximo Cupón']).dt.date
    
    ruta = 'V:/BIVSA/ADMOPER/Cierre diario/Control/Control precios al cierre ' + fa +'.xlsx' #Define el nombre y ruta del archivo.#
    control.to_excel(ruta, index=False) #Convierte y exporta la tabla total a Excel.
    print("📂 Archivo exportado exitosamente a Excel en", ruta) #Mensaje a mostrar en la consola.
    logging.info('Se genera ' + ruta) #Mensaje para el logging.

#### Tablas para enviar por mail

In [ ]:
def tablas_mail():
    tabla_mail = tabla_control() #DataFerame de la tabla de control.
    tabla_mail = tabla_mail[(tabla_mail['Días restantes']!= 0) & (tabla_mail['Días restantes']!= 1)] #Filtra para quitar las especies q cortan cupón en 0 y 1 día.  
    order = [3,4,5,6,8,10,11,12]  #Determina el orden de las columnas que necesita.
    tabla_mail = tabla_mail.iloc[:, order] #Crea tabla de principales bajas #Ordena las columnas.
    
    tabla_mail = tabla_mail.copy() #Crea una copia de la tabla para trabajar sobre ella.
    tabla_mail['Próximo Cupón'] = pd.to_datetime(tabla_mail['Próximo Cupón']).dt.strftime('%d/%m/%Y')
    tabla_mail['Próximo Cupón'] = tabla_mail['Próximo Cupón'].astype(str).replace('nan', '') #Reemplaza valores NaN.
    tabla_mail['Variación (%)'] = tabla_mail['Variación (%)'].round(2)
    tabla_mail['Criterio'] = tabla_mail['Criterio'].astype(str).replace('nan', '') #Reemplaza valores NaN.
    
    tabla_top = tabla_mail.head(10) #Crea y ordena tabla de principales subas
    tabla_low = tabla_mail.tail(10).sort_values(by='Variación (%)', ascending=False) #Crea y ordena tabla de principales subas.

    return {'top': tabla_top, 'low': tabla_low}

#### Envío de mail

In [ ]:
def envio_mail():
    #Ejecuta las funciones que generan y devuelven las tablas individuales
    tablas = tablas_mail()  #DataFerame de tablas mail
    
    top = tablas['top'] #Separa la tabla de Precios Cafci.
    top.set_index(('Variación (%)'), inplace=False)
     
    low = tablas['low'] #Separa la tabla de Precios Cafci.
    low.set_index(('Variación (%)'), inplace=False)
   
    outlook = win32.Dispatch('outlook.application')
    mail = outlook.CreateItem(0)
    mail.To = 'jose.aristi@icbc.com.ar; sergio.maugeri@icbc.com.ar; leandro.szymanski@icbc.com.ar; leonardo.lopez@icbc.com.ar; ruben.llambi@icbc.com.ar; daiana.andres@icbc.com.ar'
    mail.CC = 'alphafondosdeinversion@icbc.com.ar'
    mail.Subject = '⚠️ Precios activos del ' + fa
    mail.HTMLBody = 'Estimados:' + '<br/><br/>'
    mail.HTMLBody = mail.HTMLBody + 'Se adjunta un archivo con las cotizaciones de los activos en cartera.'+ '<br/><br/>'
    mail.HTMLBody = mail.HTMLBody + '''📉 <u>Principales bajas:</u><br/><br/>
                    {}'''.format(top.to_html(index=False)) + '<u>Nota:</u> se excluyen las especies con cupones cobrados en el día y con corte programado para el día hábil siguiente.<br/><br/>'
    mail.HTMLBody = mail.HTMLBody + '''📈 <u>Principales subas:</u><br/><br/>
                    {}'''.format(low.to_html(index=False)) + '<br/>'
    mail.HTMLBody = mail.HTMLBody + '''🧮 <u>Curvas de Renta Fija</u><br/><br/>'''
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_Badlar_TAMAR FF.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_DL ON PyME.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_DL ON.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_ON ARS BADLAR&TAMAR.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_ON CER.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_ON HD Ley Arg.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_ON HD Ley Ext.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_Tasa Fija ON.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_Soberano CER.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + f""" <html> <body> <p><img src="{'R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_Soberano HD.jpg'}"</p></body></html>"""
    mail.HTMLBody = mail.HTMLBody + 'En caso de no recibir comentarios sobre los valores adjuntos, se tomarán como válidos.'+ '<br/><br/>'      
    mail.HTMLBody = mail.HTMLBody + 'Saludos!'
    #Archivos adjuntos
    mail.Attachments.Add('V:/BIVSA/ADMOPER/Cierre diario/Control/Control precios al cierre ' + fa +'.xlsx')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_Badlar_TAMAR FF.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_DL ON PyME.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_DL ON.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_ON ARS BADLAR&TAMAR.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_ON CER.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_ON HD Ley Arg.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_ON HD Ley Ext.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_Tasa Fija ON.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_Soberano CER.jpg')
    mail.Attachments.Add('R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/Curva_Soberano HD.jpg')
    mail.Send() #Comando que envía el correo
    
    print('📨 Curvas y precios enviados por correo electrónico!') #Mensaje a mostrar en la consola.
    logging.info('Se envia e-mail a los Portfolio Managers con precios y curvas.') #Mensaje para el logging. 

#### Archivado de la información

In [ ]:
def archivado():
    origen='V:/BIVSA/ADMOPER/Cierre diario/Control/Control precios al cierre ' + fa +'.xlsx'
    destino='V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa
    shutil.copy2(origen, destino)

    ro='V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa +'/Control precios al cierre ' + fa +'.xlsx' #Define el nombre y ruta del archivo.
    win32api.SetFileAttributes(ro, win32con.FILE_ATTRIBUTE_READONLY) #Configura el archivo como sólo lectura.

    #---Back up curvas-------------------------------    
    origen_curvas='R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/outputs/'
    destino_curvas='R:/Files/User/PORTFOLI/PORFOLIO/RentaFija/Vectores/Curvas Renta Fija/Historial curvas/' + fa
    shutil.copytree(origen_curvas, destino_curvas)

    logging.info('Se realiza el backup de la información generada.') #Mensaje para el logging. 
    logging.info('Fin del proceso.') #Mensaje para el logging.

#### Función Log de cierre

In [ ]:
def cierre_log():
    shutil.copy(log_local, log_red) #Copia el log del arvhivo local a la red.
    fhandler.close() #Libera el archivo de log.
    logger.removeHandler(fhandler) #Elimina el handler
    if os.path.exists(log_local): #Elimina el archivo local.
        os.remove(log_local)
    print('✅ Información de cierre generada satisfactoriamente!') #Mensaje a mostrar en la consola.
    print('🥇Fin.')

#### Función Usuario válido

In [ ]:
def usuario_valido():
    inicio_control()
    actualizaciones()
    curvas()
    tabla_control()
    export()
    tablas_mail()
    envio_mail()
    archivado()
    cierre_log()

#### Función Usuario inválido

In [ ]:
def usuario_invalido():
    print("Acceso denegado. Usuario inválido.")
    outlook = win32.Dispatch('outlook.application')
    mail = outlook.CreateItem(0)
    mail.To = 'alphafondosdeinversion@icbc.com.ar' 
    mail.Subject = '⚠️ ATENCION: Acceso no permitido a Control de precios!!'
    mail.HTMLBody = 'Un usuario intentó acceder al archivo de control de precios al cierre.' + '<br/><br/>'
    mail.Send()

#### Ejecucion

In [ ]:
print('----Control de precios Version 3.0 - Actualizado: 20/08/2025----')

user=os.getenv('USERNAME')
user=user.lower()
print("🔑 Usuario detectado:", user)

if user in usuarios:
    descarga = input("📌¿Descargó la información necesaria:\n -VF: Cotizaciones de papeles\n -VF: Custodia RFG\n -VF: Cobro de cupones? (y/n): ")
    descarga = descarga.lower()
    if descarga =="y":
        usuario_valido()
    else:
        print('🔔 Descargue la información y vuelva a intentarlo.')
else:
    usuario_invalido()